# Lecture 5 — Forecast Uncertainty and Prediction Intervals
**ECON 417 · Business Forecasting · Southern Illinois University Edwardsville**

In Lectures 1–4 we built point forecasts — one number for each future period.  
But decisions rarely depend only on the center of the forecast.  

A portfolio manager doesn't just ask "what will the S&P 500 return next week?"  
They ask: "*What is the worst plausible outcome?* Can I afford it? Should I hedge?"

That question cannot be answered without **uncertainty quantification**.

| Section | Topic |
|---------|-------|
| 1 | Point forecast vs. prediction interval |
| 2 | Residual-based interval construction |
| 3 | Coverage vs. sharpness trade-off |
| 4 | State-dependent uncertainty — high vs. low VIX regimes |
| 5 | Scenario forecasting — bull, bear, and neutral paths |
| 6 | Asymmetric costs of forecast errors |
| 7 | Translating forecasts into actionable recommendations |

## Setup

In [ ]:
%pip install yfinance scikit-learn statsmodels matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

COLORS = {
    'train':   '#2196F3',
    'val':     '#FF9800',
    'test':    '#4CAF50',
    'ols':     '#E53935',
    'ridge':   '#7B1FA2',
    'lasso':   '#00897B',
    'neutral': '#546E7A',
    'accent':  '#F4A261',
}

GRAPH_DIR = '../Lecture 5/graph'
os.makedirs(GRAPH_DIR, exist_ok=True)

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(np.asarray(a), np.asarray(b))))

In [ ]:
# ── Download financial data (same dataset as L1–L4) ──────────────────────
DATA_SOURCE = 'synthetic (calibrated to real S&P 500 properties)'
raw = None
try:
    import yfinance as yf
    tickers = ['^GSPC', '^VIX', 'MSFT', 'AAPL', 'JPM', 'XOM']
    _raw = yf.download(tickers, start='2015-01-01', end='2024-12-31',
                       auto_adjust=True, progress=False)
    if len(_raw) > 100:
        raw = _raw
        DATA_SOURCE = 'real (yfinance)'
except Exception:
    pass

if raw is not None:
    close = raw['Close']
    sp500 = close['^GSPC']
    vix   = raw['Close']['^VIX'].reindex(sp500.index).ffill()
    msft  = close['MSFT']
    aapl  = close['AAPL']
    jpm   = close['JPM']
    xom   = close['XOM']
else:
    np.random.seed(42)
    dates = pd.date_range('2015-01-02', '2024-12-31', freq='B')
    N = len(dates)
    sp500_ret = np.random.normal(0.0003, 0.010, N)
    sp500 = pd.Series(4000 * np.exp(np.cumsum(sp500_ret)), index=dates)
    vix_vals = 18 + np.cumsum(np.random.normal(0, 0.2, N) - 0.3 * sp500_ret * 10)
    vix_vals = np.clip(vix_vals, 10, 80)
    vix = pd.Series(vix_vals, index=dates)
    def make_stock(base, beta, idio):
        r = beta * sp500_ret + np.random.normal(0, idio, N)
        return pd.Series(base * np.exp(np.cumsum(r)), index=dates)
    msft = make_stock(300, 1.2, 0.012)
    aapl = make_stock(150, 1.15, 0.013)
    jpm  = make_stock(80,  1.05, 0.011)
    xom  = make_stock(60,  0.80, 0.014)

df = pd.DataFrame({'SP500': sp500, 'VIX': vix,
                   'MSFT': msft, 'AAPL': aapl, 'JPM': jpm, 'XOM': xom}).dropna()
rets = np.log(df[['SP500','MSFT','AAPL','JPM','XOM']]).diff().dropna()
rets['VIX']     = df['VIX'].reindex(rets.index)
rets['VIX_chg'] = df['VIX'].diff().reindex(rets.index)
rets = rets.dropna()

print(f"Data source : {DATA_SOURCE}")
print(f"Observations: {len(rets):,}  ({rets.index[0].date()} → {rets.index[-1].date()})")

In [ ]:
# ── Build lagged feature matrix + 70/15/15 split ─────────────────────────
X_all = pd.DataFrame({
    'lag_SP500': rets['SP500'].shift(1),
    'lag_MSFT':  rets['MSFT'].shift(1),
    'lag_AAPL':  rets['AAPL'].shift(1),
    'lag_JPM':   rets['JPM'].shift(1),
    'lag_XOM':   rets['XOM'].shift(1),
    'VIX':       rets['VIX'].shift(1),
    'VIX_chg':   rets['VIX_chg'].shift(1),
}).dropna()
y_all = rets['SP500'].reindex(X_all.index)

T    = len(X_all)
n_tr = int(T * 0.70)
n_va = int(T * 0.15)

X_tr  = X_all.iloc[:n_tr];                y_tr  = y_all.iloc[:n_tr]
X_val = X_all.iloc[n_tr:n_tr+n_va];      y_val = y_all.iloc[n_tr:n_tr+n_va]
X_te  = X_all.iloc[n_tr+n_va:];          y_te  = y_all.iloc[n_tr+n_va:]
FEATURES = list(X_all.columns)

# ── Forecasting model: random forest (L4 champion) ───────────────────────
rf = RandomForestRegressor(n_estimators=200, max_depth=5,
                            max_features='sqrt', random_state=417)
rf.fit(X_tr, y_tr)

# Out-of-sample residuals on VALIDATION set (calibrate uncertainty honestly)
val_pred  = rf.predict(X_val)
val_resid = y_val.values - val_pred

# Test-set point forecasts
test_pred = rf.predict(X_te)

print(f"Train : {len(y_tr):4d} obs  | Val: {len(y_val):4d} obs  | Test: {len(y_te):4d} obs")
print(f"Test RMSE (point forecast): {rmse(y_te, test_pred):.6f}")
print(f"Validation residual  mean: {np.mean(val_resid):.6f}  (bias check)")
print(f"Validation residual   std: {np.std(val_resid):.6f}")

---
## Section 1 — Point Forecast vs. Prediction Interval

### Why one number is not enough

A point forecast says: *"I expect the S&P 500 to return +0.3% tomorrow."*  
A prediction interval says: *"I expect the return to fall between −1.4% and +2.0% with 80% probability."*

For an equity portfolio manager, these are very different inputs:  
- The **point forecast** suggests holding steady.  
- The **interval** reveals that a −1.4% day is within the 80% range — meaningful for a leveraged position.

**Rule:** A forecast without uncertainty is an incomplete forecast.

In [ ]:
# ── Example 5.1 — Point forecast with 80% prediction interval ────────────

q80 = np.quantile(val_resid, [0.10, 0.90])   # empirical 80% interval from OOS residuals
q95 = np.quantile(val_resid, [0.025, 0.975]) # empirical 95% interval

lo80 = test_pred + q80[0]
hi80 = test_pred + q80[1]
lo95 = test_pred + q95[0]
hi95 = test_pred + q95[1]

# Show the full test period
W = len(y_te)

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(y_te.index, lo95, hi95, color=COLORS['train'], alpha=0.12, label='95% interval')
ax.fill_between(y_te.index, lo80, hi80, color=COLORS['train'], alpha=0.28, label='80% interval')
ax.plot(y_te.index, y_te.values, color=COLORS['neutral'], lw=1.2, alpha=0.9, label='Actual SP500 return')
ax.plot(y_te.index, test_pred,   color=COLORS['train'],   lw=1.8, label='Point forecast', zorder=4)
ax.axhline(0, color='black', lw=0.7, ls='--', alpha=0.4)

# Mark days where actual fell outside 80% interval
miss80 = (y_te.values < lo80) | (y_te.values > hi80)
ax.scatter(y_te.index[miss80], y_te.values[miss80],
           color=COLORS['ols'], s=18, zorder=5, label=f'Outside 80% interval ({miss80.mean()*100:.0f}% of days)')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('S&P 500 Log Return', fontsize=11)
ax.set_title('Example 5.1 — Point Forecast With 80% and 95% Prediction Intervals\n'
             'Intervals widen the risk picture beyond the center estimate',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_5_1_point_and_interval.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"80% interval: [{q80[0]*100:.2f}%, {q80[1]*100:.2f}%]")
print(f"95% interval: [{q95[0]*100:.2f}%, {q95[1]*100:.2f}%]")

**What the interval tells a portfolio manager:**  
Even on an average day, the 80% interval spans roughly ±1–2%.  
For a fund with 10× leverage, a lower bound of −1.5% means a −15% portfolio move is within the plausible range.  
The point forecast alone would never reveal this risk.

---
## Section 2 — Residual-Based Interval Construction

### Where intervals come from

The simplest approach:

$$\hat{y}_{t+1} \pm z_{\alpha/2} \cdot \hat{\sigma}_e$$

This formula assumes forecast errors are *normally distributed*.  
Financial returns are well-known to be **fat-tailed** — extreme days happen far more often than a normal distribution predicts.  
Using the normal formula would *understate* the width of the tails where risk actually lives.

**Better approach — empirical quantiles:**
$$\text{Lower} = \hat{y} + Q_{\alpha/2}(\text{OOS residuals}), \quad \text{Upper} = \hat{y} + Q_{1-\alpha/2}(\text{OOS residuals})$$

The quantiles are read directly from the actual residual distribution, respecting its real shape — skew, kurtosis, and fat tails included.  
We always use **out-of-sample** (validation) residuals, not in-sample residuals, to avoid optimistic bias.

In [ ]:
# ── Example 5.2 — Residual distribution and empirical quantiles ───────────

from scipy import stats as scipy_stats

# Fit a normal distribution for comparison
mu_r, sigma_r = np.mean(val_resid), np.std(val_resid)
kurt = scipy_stats.kurtosis(val_resid)   # excess kurtosis (normal = 0)

xg = np.linspace(val_resid.min() * 1.1, val_resid.max() * 1.1, 300)
normal_pdf = scipy_stats.norm.pdf(xg, mu_r, sigma_r)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: histogram + normal overlay
ax = axes[0]
ax.hist(val_resid, bins=40, density=True, color=COLORS['train'], alpha=0.7,
        edgecolor='white', label='OOS residual distribution')
ax.plot(xg, normal_pdf, color=COLORS['ols'], lw=2.5, ls='--',
        label=f'Normal fit (μ={mu_r:.4f}, σ={sigma_r:.4f})')
for pct, q_val, label in [
    (5,  np.percentile(val_resid, 5),  '5th percentile'),
    (10, np.percentile(val_resid, 10), '10th percentile'),
    (90, np.percentile(val_resid, 90), '90th percentile'),
    (95, np.percentile(val_resid, 95), '95th percentile'),
]:
    ax.axvline(q_val, color=COLORS['val'], lw=1.5, ls=':', alpha=0.8)
    ax.text(q_val, ax.get_ylim()[1] * 0.85 if pct < 50 else ax.get_ylim()[1] * 0.75,
            f'{q_val*100:.1f}%', fontsize=8, color=COLORS['val'], ha='center')
ax.set_xlabel('Validation Residual (return units)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('OOS Residual Distribution\nFat tails: normal fit underestimates extreme events', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# Right: QQ-plot to show fat tails
ax = axes[1]
(osm, osr), (slope, intercept, _) = scipy_stats.probplot(val_resid, dist='norm')
ax.scatter(osm, osr, color=COLORS['neutral'], s=8, alpha=0.5, label='Residuals')
line_x = np.array([min(osm), max(osm)])
ax.plot(line_x, slope * line_x + intercept, color=COLORS['ols'], lw=2,
        ls='--', label='Normal reference line')
ax.set_xlabel('Theoretical Normal Quantiles', fontsize=11)
ax.set_ylabel('Observed Residual Quantiles', fontsize=11)
ax.set_title(f'Normal Q-Q Plot\nFat tails visible at both ends (excess kurtosis = {kurt:.2f})', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

fig.suptitle('Example 5.2 — Residual Distribution: Why We Use Empirical Quantiles\n'
             'Normal formula understates tail risk in financial return forecasting',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_5_2_residual_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Excess kurtosis of residuals: {kurt:.3f}  (normal = 0; larger = fatter tails)")
print(f"Normal formula 10th pct: {(mu_r + scipy_stats.norm.ppf(0.10) * sigma_r)*100:.2f}%")
print(f"Empirical 10th pct:       {np.percentile(val_resid, 10)*100:.2f}%")

**Key observation:**  
The Q-Q plot's tails curve away from the reference line — the hallmark of fat tails.  
The empirical 10th percentile is *further from zero* than what the normal formula predicts.  
Using the normal formula for a risk management system would underestimate the frequency of bad days.

---
## Section 3 — Coverage vs. Sharpness

A good prediction interval balances two competing goals:

- **Coverage:** The fraction of realized outcomes that fall inside the interval should match the stated level (e.g., 80% interval should contain 80% of actual returns).
- **Sharpness:** The interval should be as *narrow* as possible — wide intervals that contain everything are useless for decisions.

**The trade-off:** Widen the interval and coverage improves; narrow it and decisions become crisper but the model fails more often.  
The right width depends on the cost of being wrong vs. the value of precision.

In [ ]:
# ── Example 5.3 — Coverage vs. sharpness for different interval widths ────

interval_specs = [
    ('50%',  0.25,  0.75),
    ('80%',  0.10,  0.90),
    ('95%',  0.025, 0.975),
]

results_cov = []
for name, lo_p, hi_p in interval_specs:
    q_lo = np.quantile(val_resid, lo_p)
    q_hi = np.quantile(val_resid, hi_p)
    lo = test_pred + q_lo
    hi = test_pred + q_hi
    coverage = np.mean((y_te.values >= lo) & (y_te.values <= hi))
    avg_width = np.mean(hi - lo) * 100  # in percentage points
    results_cov.append({'Interval': name, 'Coverage': coverage,
                         'Avg Width (pp)': avg_width})

cov_df = pd.DataFrame(results_cov)
print("Coverage vs. Sharpness on Test Set:")
print(cov_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
bar_colors = [COLORS['lasso'], COLORS['train'], COLORS['ridge']]

ax = axes[0]
bars = ax.bar(cov_df['Interval'], cov_df['Coverage'], color=bar_colors, edgecolor='white', width=0.5)
ax.axhline(0.50, color=COLORS['lasso'], lw=1.5, ls='--', alpha=0.7, label='Target 50%')
ax.axhline(0.80, color=COLORS['train'], lw=1.5, ls='--', alpha=0.7, label='Target 80%')
ax.axhline(0.95, color=COLORS['ridge'], lw=1.5, ls='--', alpha=0.7, label='Target 95%')
for bar, val in zip(bars, cov_df['Coverage']):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.12)
ax.set_ylabel('Empirical Coverage', fontsize=11)
ax.set_title('Empirical Coverage\n(horizontal lines = stated targets)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

ax = axes[1]
bars = ax.bar(cov_df['Interval'], cov_df['Avg Width (pp)'], color=bar_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, cov_df['Avg Width (pp)']):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.2f} pp', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Average Interval Width (percentage points)', fontsize=11)
ax.set_title('Interval Sharpness\n(narrower = more actionable)', fontsize=12, fontweight='bold')

fig.suptitle('Example 5.3 — Coverage vs. Sharpness: The Unavoidable Trade-Off\n'
             'Wider intervals are more honest; narrower intervals guide decisions more sharply',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_5_3_coverage_width_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4 — State-Dependent Uncertainty: High vs. Low VIX Regimes

### Forecast errors are not constant across all market conditions

When the VIX is low (calm markets), daily returns are small and predictable.  
When the VIX is high (stressed markets), returns are large and chaotic — and forecast errors are correspondingly wider.

This is **heteroskedasticity** in the forecast error: the variance of errors changes depending on market conditions.  
A single fixed interval width can systematically **understate risk** in high-VIX periods
and **overstate uncertainty** in calm periods — both of which mislead decisions.

**Financial interpretation:** A portfolio manager using a fixed ±1.5% daily interval  
would be well-calibrated during normal markets but dangerously under-hedged during a crisis.

In [ ]:
# ── Example 5.4 — State-dependent uncertainty by VIX regime ───────────────

# Categorise validation periods by VIX regime
val_vix = X_val['VIX'].values

low_vix    = val_vix < 15
medium_vix = (val_vix >= 15) & (val_vix < 25)
high_vix   = val_vix >= 25

regime_data = {
    'Calm\n(VIX < 15)':     val_resid[low_vix],
    'Normal\n(15 ≤ VIX < 25)': val_resid[medium_vix],
    'Stressed\n(VIX ≥ 25)': val_resid[high_vix],
}

print("Error statistics by VIX regime:")
for label, res in regime_data.items():
    print(f"  {label.replace(chr(10), ' '):30s}  n={len(res):4d}  "
          f"std={np.std(res):.5f}  "
          f"[10th pct={np.percentile(res,10)*100:.2f}%, 90th pct={np.percentile(res,90)*100:.2f}%]")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: boxplot of residuals by regime
ax = axes[0]
bp_data   = list(regime_data.values())
bp_labels = list(regime_data.keys())
bp = ax.boxplot(bp_data, patch_artist=True, labels=bp_labels,
                 showfliers=True, flierprops=dict(marker='.', markersize=3, alpha=0.4))
bp_colors = [COLORS['test'], COLORS['neutral'], COLORS['ols']]
for patch, col in zip(bp['boxes'], bp_colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.4)
ax.set_ylabel('Forecast Error (return units)', fontsize=11)
ax.set_title('Forecast Errors by VIX Regime\nBox shows IQR; whiskers extend to 1.5×IQR', fontsize=12, fontweight='bold')

# Right: interval width comparison
ax = axes[1]
regime_widths = {}
for label, res in regime_data.items():
    if len(res) > 5:
        w = (np.percentile(res, 90) - np.percentile(res, 10)) * 100
        regime_widths[label.replace('\n', ' ')] = w

bars = ax.bar(list(regime_widths.keys()), list(regime_widths.values()),
               color=bp_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, regime_widths.values()):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{val:.2f}pp', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('80% Interval Width (percentage points)', fontsize=11)
ax.set_title('80% Interval Width by VIX Regime\nUncertainty is much larger in stressed markets', fontsize=12, fontweight='bold')

fig.suptitle('Example 5.4 — State-Dependent Uncertainty: Errors Widen in High-VIX Periods\n'
             'A fixed interval width systematically misstates risk in different market conditions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_5_4_state_dependent_uncertainty.png', dpi=150, bbox_inches='tight')
plt.show()

**Key takeaway:**  
The 80% interval during a stressed (high-VIX) period is substantially wider than during a calm period.  
A risk system that uses one fixed interval ignores this regime difference — and will fail exactly when it matters most (during crises).

**Practical response:** Use *state-conditional intervals* — calibrate interval widths separately by VIX regime.  
This is an extension beyond what we implement today, but the lesson is clear: heteroskedasticity is not a nuisance, it's risk information.

---
## Section 5 — Scenario Forecasting: Bull, Bear, and Neutral

### Asking "what if" instead of predicting one path

A scenario forecast replaces the single predicted path with multiple paths under different assumptions.  
For financial forecasting, the scenarios correspond to plausible short-term market states:

| Scenario | VIX trend | Momentum | Interpretation |
|----------|-----------|----------|-----------|
| **Bull** | VIX declining | Positive momentum | Risk-on: low fear, upward trend |
| **Neutral** | VIX flat | Zero momentum | Sideways drift |
| **Bear** | VIX rising | Negative momentum | Risk-off: fear rising, downward trend |

The *gap* between the bull and bear paths is often more useful for decision-making than any single forecast number.

In [ ]:
# ── Example 5.5 — Scenario forecasting: bull, neutral, bear ──────────────

N_FWD = 15  # 15 trading days (~3 weeks)

# Start from the last observed feature values
last = X_te.iloc[-1].copy()

def build_scenario(vix_daily_chg, momentum_daily, n=N_FWD):
    """Construct a sequence of feature rows for a scenario."""
    rows = []
    state = last.copy().to_dict()
    for _ in range(n):
        rows.append(state.copy())
        # Update state for next step
        state['VIX']      = max(10.0, state['VIX'] + vix_daily_chg)
        state['VIX_chg']  = vix_daily_chg
        state['lag_SP500'] = momentum_daily
        # Other stock lags drift toward zero gradually
        for col in ['lag_MSFT','lag_AAPL','lag_JPM','lag_XOM']:
            state[col] = state[col] * 0.5 + momentum_daily * 0.5
    return pd.DataFrame(rows, columns=FEATURES)

bull_X    = build_scenario(vix_daily_chg=-0.4, momentum_daily=+0.007)
neutral_X = build_scenario(vix_daily_chg=0.0,  momentum_daily=0.0)
bear_X    = build_scenario(vix_daily_chg=+1.2, momentum_daily=-0.010)

bull_fc    = rf.predict(bull_X)
neutral_fc = rf.predict(neutral_X)
bear_fc    = rf.predict(bear_X)

# Cumulative return for each scenario
cum_bull    = np.cumsum(bull_fc) * 100
cum_neutral = np.cumsum(neutral_fc) * 100
cum_bear    = np.cumsum(bear_fc) * 100

days = np.arange(1, N_FWD + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: daily forecast per scenario
ax = axes[0]
ax.plot(days, bull_fc * 100,    marker='o', ms=5, color=COLORS['test'],  lw=2.2, label='Bull: VIX↓, momentum+')
ax.plot(days, neutral_fc * 100, marker='o', ms=5, color=COLORS['neutral'], lw=2.2, label='Neutral: flat')
ax.plot(days, bear_fc * 100,    marker='o', ms=5, color=COLORS['ols'],   lw=2.2, label='Bear: VIX↑, momentum−')
ax.axhline(0, color='black', lw=0.7, ls='--', alpha=0.4)
ax.set_xlabel('Trading Day Ahead', fontsize=11)
ax.set_ylabel('Predicted Daily Return (%)', fontsize=11)
ax.set_title('Daily Predicted Returns by Scenario', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Right: cumulative return
ax = axes[1]
ax.fill_between(days, cum_bear, cum_bull, alpha=0.18, color=COLORS['train'], label='Scenario spread')
ax.plot(days, cum_bull,    marker='o', ms=5, color=COLORS['test'],    lw=2.2, label='Bull cumulative')
ax.plot(days, cum_neutral, marker='o', ms=5, color=COLORS['neutral'],  lw=2.2, label='Neutral cumulative')
ax.plot(days, cum_bear,    marker='o', ms=5, color=COLORS['ols'],     lw=2.2, label='Bear cumulative')
ax.axhline(0, color='black', lw=0.7, ls='--', alpha=0.4)
ax.set_xlabel('Trading Day Ahead', fontsize=11)
ax.set_ylabel('Cumulative Return (%)', fontsize=11)
ax.set_title(f'Cumulative 15-Day Horizon\nSpread = {cum_bull[-1] - cum_bear[-1]:.1f} pp bull-minus-bear', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

fig.suptitle('Example 5.5 — Scenario Forecasts: Bull, Neutral, and Bear Paths\n'
             'The scenario spread matters more than any single number for portfolio decisions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_5_5_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"15-day cumulative return: Bull = {cum_bull[-1]:.2f}%  |  Neutral = {cum_neutral[-1]:.2f}%  |  Bear = {cum_bear[-1]:.2f}%")

---
## Section 6 — Asymmetric Costs of Forecast Errors

### Not all forecast errors are equally bad

Standard metrics like RMSE and MAE treat over-forecasting and under-forecasting symmetrically.  
But in most real financial decisions, the two types of error have very different consequences:

**Example — a risk manager at an asset management firm:**
- **Under-forecasting returns** (thinking market is flat when it rallies): the fund misses gains, slightly underperforms the benchmark → moderate cost.
- **Over-forecasting returns** (thinking market will rally when it crashes): the fund is over-invested in a falling market → large losses, regulatory scrutiny, possible fund closure → high cost.

**Asymmetric loss framework:**

Let $e_t = y_t - \hat{y}_t$ be the forecast error.

$$\text{Cost}(e_t) = \begin{cases}
c_{\text{under}} \cdot |e_t| & \text{if } e_t > 0 \quad (\text{actual > forecast: under-forecast})\\
c_{\text{over}}  \cdot |e_t| & \text{if } e_t < 0 \quad (\text{actual < forecast: over-forecast})
\end{cases}$$

**Key insight:** When $c_{\text{over}} > c_{\text{under}}$, the optimal strategy is to forecast conservatively (lower predicted return) — the model should deliberately plan to the *pessimistic side* of the distribution.

In [ ]:
# ── Example 5.6 — Asymmetric cost framework ───────────────────────────────

# Cost parameters (risk manager scenario)
# Over-forecasting (too optimistic) is 3× more costly than under-forecasting
C_UNDER = 1.0   # cost per unit of under-forecast error
C_OVER  = 3.0   # cost per unit of over-forecast error

errors = y_te.values - test_pred

def asymmetric_cost(errors, c_under, c_over):
    return np.where(errors > 0, c_under * np.abs(errors),
                                c_over  * np.abs(errors))

cost_symmetric   = np.abs(errors)   # MAE
cost_asymmetric  = asymmetric_cost(errors, C_UNDER, C_OVER)

# Symmetric cost assumes we use the median (50th pct) forecast
# Optimal asymmetric: shift forecast to the c_over/(c_over+c_under) quantile
# This minimises expected asymmetric cost
optimal_quantile = C_OVER / (C_OVER + C_UNDER)   # = 3/(3+1) = 0.75
conservative_adj = np.quantile(val_resid, 1 - optimal_quantile)  # negative shift
conservative_pred = test_pred + conservative_adj

errors_conservative = y_te.values - conservative_pred
cost_conservative   = asymmetric_cost(errors_conservative, C_UNDER, C_OVER)

print(f"Cost parameters: c_under = {C_UNDER}, c_over = {C_OVER}")
print(f"Optimal quantile to forecast at: {optimal_quantile:.0%}  (25th pct of return distribution)")
print(f"Conservative adjustment:         {conservative_adj*100:.3f}% per day")
print()
print(f"Mean daily symmetric cost (MAE): {np.mean(cost_symmetric):.6f}")
print(f"Mean daily asymmetric cost — point forecast:    {np.mean(cost_asymmetric):.6f}")
print(f"Mean daily asymmetric cost — conservative pred: {np.mean(cost_conservative):.6f}")
print(f"Cost reduction: {(np.mean(cost_asymmetric)-np.mean(cost_conservative))/np.mean(cost_asymmetric)*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
xr = np.linspace(-0.04, 0.04, 400)
symm_cost = np.abs(xr)
asym_cost = np.where(xr > 0, C_UNDER * np.abs(xr), C_OVER * np.abs(xr))
ax.plot(xr * 100, symm_cost * 100,  color=COLORS['neutral'], lw=2.5, ls='--', label='Symmetric (MAE)')
ax.plot(xr * 100, asym_cost * 100,  color=COLORS['ols'],     lw=2.5,           label=f'Asymmetric (c_over={C_OVER}×)')
ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.4)
ax.fill_between(xr * 100, asym_cost * 100, symm_cost * 100,
                 where=(xr < 0), alpha=0.2, color=COLORS['ols'],
                 label='Extra cost for over-forecast')
ax.set_xlabel('Forecast Error (actual − forecast, pp)', fontsize=11)
ax.set_ylabel('Cost (pp)', fontsize=11)
ax.set_title(f'Asymmetric Loss Function\nc_under={C_UNDER}  |  c_over={C_OVER}  →  plan to {optimal_quantile:.0%} quantile', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

ax = axes[1]
bars = ax.bar(['Symmetric\n(MAE)', 'Asymmetric\n(point)', 'Asymmetric\n(conservative)'],
               [np.mean(cost_symmetric), np.mean(cost_asymmetric), np.mean(cost_conservative)],
               color=[COLORS['neutral'], COLORS['ols'], COLORS['test']], edgecolor='white', width=0.5)
for bar, val in zip(bars, [np.mean(cost_symmetric), np.mean(cost_asymmetric), np.mean(cost_conservative)]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.01,
            f'{val:.5f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean Daily Asymmetric Cost', fontsize=11)
ax.set_title('Cost Comparison\nConservative forecast minimises total asymmetric cost', fontsize=12, fontweight='bold')

fig.suptitle('Example 5.6 — Asymmetric Costs: Risk Manager Scenario (c_over > c_under)\n'
             'Over-forecasting returns is more costly → optimal plan is at the 25th percentile',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_5_6_asymmetric_cost.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — Translating Forecasts Into Actionable Recommendations

### The forecast is not the final product — the decision is

A forecast answers: *What will happen?*  
A recommendation answers: *Given what might happen, what should we do?*

**Three-step translation template:**

1. **State the forecast.**  
   *"The model predicts an S&P 500 return of +0.25% tomorrow, with an 80% interval of [−1.1%, +1.6%]."*

2. **State the relevant threshold or constraint.**  
   *"Our fund's daily loss limit is −1.0%. The 80% lower bound of −1.1% breaches this threshold."*

3. **State the recommendation.**  
   *"We recommend reducing equity exposure by 20% before tomorrow's open as a precautionary hedge."*

In [ ]:
# ── Example 5.7 — Forecast-to-decision: portfolio exposure ────────────────

# Suppose we have a daily loss limit of -1.5%
# If the 80% lower bound breaches the limit, we flag for reduced exposure
LOSS_LIMIT = -0.015   # -1.5% daily loss limit

flag_days = lo80 < LOSS_LIMIT
flag_count = flag_days.sum()

# Show the last 60 test days for clarity
W_plot = min(60, len(y_te))
idx = y_te.index[-W_plot:]
te_slice   = y_te.values[-W_plot:]
pred_slice = test_pred[-W_plot:]
lo_slice   = lo80[-W_plot:]
hi_slice   = hi80[-W_plot:]
flag_slice = flag_days[-W_plot:]

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(idx, lo_slice, hi_slice, color=COLORS['train'], alpha=0.25, label='80% interval')
ax.plot(idx, te_slice,   color=COLORS['neutral'], lw=1.3, label='Actual return')
ax.plot(idx, pred_slice, color=COLORS['train'],   lw=1.8, label='Point forecast', zorder=4)
ax.axhline(LOSS_LIMIT, color=COLORS['ols'], lw=2.0, ls='--',
           label=f'Daily loss limit ({LOSS_LIMIT*100:.1f}%)', zorder=3)

# Highlight flagged days
for i, (date, flagged) in enumerate(zip(idx, flag_slice)):
    if flagged:
        ax.axvspan(date, date, alpha=0.25, color=COLORS['ols'], lw=0)

# Add one summary span label
if flag_slice.any():
    flag_idx = np.where(flag_slice)[0][0]
    ax.annotate('⚠ Reduce exposure\n(80% LB < loss limit)',
                xy=(idx[flag_idx], LOSS_LIMIT),
                xytext=(idx[flag_idx], LOSS_LIMIT - 0.008),
                fontsize=9, color=COLORS['ols'], fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=COLORS['ols']))

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('S&P 500 Log Return', fontsize=11)
ax.set_title('Example 5.7 — Forecast-to-Decision: Portfolio Exposure Management\n'
             f'Red shading: 80% lower bound below loss limit → {flag_count} flag days in full test set',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_5_7_forecast_decision.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n--- Three-Step Decision Translation ---")
last_pred  = test_pred[-1]
last_lo80  = lo80[-1]
last_hi80  = hi80[-1]
flagged    = last_lo80 < LOSS_LIMIT
print(f"1. FORECAST: Point = {last_pred*100:+.3f}%  |  80% interval = [{last_lo80*100:.3f}%, {last_hi80*100:.3f}%]")
print(f"2. THRESHOLD: Daily loss limit = {LOSS_LIMIT*100:.1f}%")
print(f"3. RECOMMENDATION: {'⚠ Lower equity exposure (80% LB breaches loss limit)' if flagged else '✓ Hold position (80% LB within loss limit)'}")

---
## Interview Preparation

**"So what would you recommend based on this forecast?"**  
> "The point forecast suggests a small positive return, but the 80% interval extends down to −1.1%.  
> Given that our fund's daily loss limit is −1.0%, the lower bound of the interval breaches that threshold.  
> I would recommend reducing equity exposure by 20% as a precautionary hedge before the open,  
> rather than taking the full position based on the point forecast alone."

**"Which type of error is more costly here?"**  
> "In this risk management context, over-forecasting returns is more costly.  
> If we forecast a rally and the market crashes, we face large losses, margin calls, and regulatory exposure.  
> The asymmetric cost structure with $c_{\text{over}} = 3\times c_{\text{under}}$ means the optimal plan  
> is at the 25th percentile of the return distribution — deliberately conservative."

**"How does this forecast change the decision?"**  
> "Without the interval, the manager sees only a positive point forecast and holds the full position.  
> With the interval, we can identify days where the lower bound crosses the loss threshold  
> and trigger a partial hedge automatically.  
> That reduces tail losses on bad days while preserving most of the upside — which is exactly what risk management is for."